In [ ]:
# | default_exp features.fsd_genomewide

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()


In [ ]:
# | export
class FSDEvaluator(FeatureEvaluator):
    """Extracts normalized densities for all fragment size buckets."""

    name = "FsdGenomewide"
    source_file = ".FSD.parquet"
    tier = 1
    category = "fragmentation"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted

            cols = set(df.columns)

            if "total" not in cols:
                log.warning("fsd_gw_missing_total_col")
                return extracted

            # FSD has multiple rows for groups (e.g. per-chromosome or total), we will just aggregate globally first
            total_reads = df["total"].sum()
            if total_reads == 0:
                return extracted

            for c in df.columns:
                if c not in ["region", "total", "chrom"]:
                    bucket_sum = df[c].sum()
                    extracted[f"fsd_gw_density_{c}"] = float(bucket_sum / total_reads)

            # Special 143/166 proxy
            # 143 falls in "140-144", 166 falls in "165-169"
            if "140-144" in cols and "165-169" in cols:
                v143 = df["140-144"].sum()
                v166 = df["165-169"].sum()
                if v166 > 0:
                    extracted["fsd_gw_143_166_ratio"] = float(v143 / v166)

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
evaluator = FSDEvaluator()
df_test = pd.DataFrame({"140-144": [200], "165-169": [500], "total": [2000]})
res = evaluator.extract(df_test)
assert "fsd_gw_density_140-144" in res
assert res["fsd_gw_density_140-144"] == 0.1
assert res["fsd_gw_143_166_ratio"] == 0.4